# Collect Historical Solar Performance Data
#### For Resource Performance Draw

### Implementation

#### Draw Performance Data

In [1]:
# Imports 
import pandas as pd 
import numpy as np
import h5py


In [ ]:
# Read HDF5 File
filepath = "upv_county/upv-reference_county.h5"

with h5py.File(filepath, "r") as f:
    cf = f["cf"][:]          # NumPy array
    index = f["index"][:]    # time labels
    columns = f["columns"][:]  # county labels


In [ ]:
# Fix data types
index = index.astype(str)
columns = columns.astype(str)

In [ ]:
# Assign to DataFrame
df = pd.DataFrame(cf, index=index, columns=columns)


In [ ]:
# Check Dataframe 
print(df.head())
print(df.shape)
print(df.index[:5])

   2_p01001  3_p01001  2_p01003  3_p01003  2_p01005  3_p01005  2_p01007  \
1       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0       0.0       0.0       0.0       0.0   
5       0.0       0.0       0.0       0.0       0.0       0.0       0.0   

   2_p01009  2_p01011  2_p01013  ...  4_p56035  3_p56037  4_p56037  1_p56039  \
1       0.0       0.0       0.0  ...       0.0       0.0       0.0       0.0   
2       0.0       0.0       0.0  ...       0.0       0.0       0.0       0.0   
3       0.0       0.0       0.0  ...       0.0       0.0       0.0       0.0   
4       0.0       0.0       0.0  ...       0.0       0.0       0.0       0.0   
5       0.0       0.0       0.0  ...       0.0       0.0       0.0       0.0   

   2_p56039  3_p56041  4_p56041  2_p56043  3_p56043  3_p56045  
1   

C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Pack into multiindex dataframe 
cols = df.columns.to_series()

meta = cols.str.extract(
    r"(?P<class>\d+)_p(?P<fips>\d+)"
)

meta["class"] = meta["class"].astype(int)
meta["fips"] = meta["fips"].astype(str).str.zfill(5)

df.columns = pd.MultiIndex.from_frame(meta)
df.columns.names = ["class", "fips"]

county_cf = df.groupby(level="fips", axis=1).mean()



In [ ]:
# Get FIPS translator
fips_translate_filename = "state_and_county_fips_master.csv"

fips_translator = pd.read_csv(fips_translate_filename, dtype={"fips": str})
fips_translator['fips'] = fips_translator['fips'].apply(lambda x: str(x).zfill(5))

print(fips_translator.head())

    fips            name state
0  00000   UNITED STATES   NaN
1  01000         ALABAMA   NaN
2  01001  Autauga County    AL
3  01003  Baldwin County    AL
4  01005  Barbour County    AL


In [10]:
# Step 1: Create a mapping from fips number to "State_County"
fips_map = fips_translator.set_index('fips').apply(lambda row: f"{row['state']}_{row['name']}", axis=1).to_dict()

new_columns = pd.MultiIndex.from_tuples(
    [(cls, fips_map.get(fips, fips)) for cls, fips in df.columns],
    names=df.columns.names
)

df.columns = new_columns



In [ ]:
# Cheeck output
print(df.head())

class                 2                 3                 2                 3  \
fips  AL_Autauga County AL_Autauga County AL_Baldwin County AL_Baldwin County   
1                   0.0               0.0               0.0               0.0   
2                   0.0               0.0               0.0               0.0   
3                   0.0               0.0               0.0               0.0   
4                   0.0               0.0               0.0               0.0   
5                   0.0               0.0               0.0               0.0   

class                 2                 3              2                   \
fips  AL_Barbour County AL_Barbour County AL_Bibb County AL_Blount County   
1                   0.0               0.0            0.0              0.0   
2                   0.0               0.0            0.0              0.0   
3                   0.0               0.0            0.0              0.0   
4                   0.0               0.0      

C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Grab only PJM States
PJM_states = ['PA', 'NJ', 'MD', 'DC', 'DE', 'IL', 'WV', 'VA', 'OH', 'IN', 'MI', 'KY', 'NC']  # added MI, KY, NC as commonly in PJM


# 2️⃣ Boolean mask for columns where the state is in PJM_states
mask = df.columns.get_level_values(1).str[:2].isin(PJM_states)

# 3️⃣ Filter columns
PJM_df = df.loc[:, mask]

PJM_frame = PJM_df.mean(axis=1).to_frame(name='pjm_avg')



In [ ]:
# Add "day" column
PJM_frame.index = PJM_frame.index.astype(int)

PJM_frame['day'] = PJM_frame.index // 24

#### Add weather data

In [ ]:
# Draw historic weather data
weather = pd.read_csv("solar_weather_data.csv")
thi_series = weather['system_max_thi']

PJM_frame['thi'] = thi_series

In [ ]:
# Test output
print(PJM_frame.head())


       pjm_avg   day        thi
1          0.0     0  57.432200
2          0.0     0  56.804000
3          0.0     0  56.174000
4          0.0     0  57.223400
5          0.0     0  58.605589
...        ...   ...        ...
61316      0.0  2554  62.255973
61317      0.0  2554  62.817728
61318      0.0  2554  60.487492
61319      0.0  2554  56.683400
61320      0.0  2555  53.083397

[61320 rows x 3 columns]


C:\Users\henry\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Save to CSV (because I was at a stopping point, this isn't strictly necessary)
PJM_frame.to_csv("pjm_solar_thi_data.csv")

In [ ]:
# Read back out of dataframe
solar_performance_dataframe = pd.read_csv("pjm_solar_thi_data.csv", index_col=0)

In [ ]:
# Read in weather data 
weather_data = pd.read_csv("solar_weather_data.csv")

In [ ]:
# Add datetime column and specify season. Get rid of "date" column. 
weather_date = weather_data['date'] 
solar_performance_dataframe['datetime'] = weather_date
cols_to_keep = ['datetime', 'pjm_avg', 'thi']
solar_performance_dataframe = solar_performance_dataframe[cols_to_keep]

solar_performance_dataframe["datetime"] = pd.to_datetime(
    solar_performance_dataframe["datetime"]
)

solar_performance_dataframe["season"] = np.where(
    solar_performance_dataframe["datetime"].dt.month.isin([10, 11, 12, 1, 2, 3]),
    "winter",
    "summer"
)

print(solar_performance_dataframe.head())



                   datetime  pjm_avg        thi  season
1 2007-01-01 01:00:00+00:00      0.0  57.432200  winter
2 2007-01-01 02:00:00+00:00      0.0  56.804000  winter
3 2007-01-01 03:00:00+00:00      0.0  56.174000  winter
4 2007-01-01 04:00:00+00:00      0.0  57.223400  winter
5 2007-01-01 05:00:00+00:00      0.0  58.605589  winter


In [ ]:
# Create better date column 
solar_performance_dataframe['date'] = solar_performance_dataframe['datetime'].dt.date

In [ ]:
# Create weather bins based on daily THI extremes 
daily_thi = (
    solar_performance_dataframe.groupby(["date", "season"])["thi"]
      .agg(
          thi_extreme=lambda x: x.max() if x.name[1] == "summer" else x.min()
      )
      .reset_index()
)

bin_edges = np.arange(daily_thi["thi_extreme"].min() // 5 * 5,
                      daily_thi["thi_extreme"].max() + 5,
                      5)

daily_thi["thi_bin"] = pd.cut(
    daily_thi["thi_extreme"],
    bins=bin_edges
)

In [ ]:
# Create daily 24-hour resource performance profiles 
daily_profiles = (
    solar_performance_dataframe.sort_values("datetime")
      .groupby("date")["pjm_avg"]
      .apply(lambda x: x.values if len(x) == 24 else None)
      .reset_index(name="solar_profile")
)
daily_profiles = daily_profiles.dropna()


In [ ]:
# Merge into readable format 
daily = daily_thi.merge(daily_profiles, on="date", how="inner")

In [45]:
daily.head()


,date,season,thi_extreme,thi_bin,solar_profile
0,2007-01-02,winter,44.173400,"(40.0, 45.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2e-06, 0..."
1,2007-01-03,winter,37.693400,"(35.0, 40.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2e-06, 0.0..."
2,2007-01-04,winter,49.753400,"(45.0, 50.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4e-07, 0.0..."
3,2007-01-05,winter,58.115310,"(55.0, 60.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.006..."
4,2007-01-06,winter,63.765006,"(60.0, 65.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.046..."


In [ ]:
# Unpack format, save to CSV
long = (
    daily
    .explode("solar_profile")
    .assign(hour=lambda x: x.groupby("date").cumcount())
)

long.to_csv("daily_solar_profiles_long.csv", index=False)

print(long)

            date  season  thi_extreme       thi_bin solar_profile  hour
0     2007-01-02  winter      44.1734  (40.0, 45.0]           0.0     0
0     2007-01-02  winter      44.1734  (40.0, 45.0]           0.0     1
0     2007-01-02  winter      44.1734  (40.0, 45.0]           0.0     2
0     2007-01-02  winter      44.1734  (40.0, 45.0]           0.0     3
0     2007-01-02  winter      44.1734  (40.0, 45.0]           0.0     4
...          ...     ...          ...           ...           ...   ...
2553  2013-12-29  winter      42.0134  (40.0, 45.0]           0.0    19
2553  2013-12-29  winter      42.0134  (40.0, 45.0]           0.0    20
2553  2013-12-29  winter      42.0134  (40.0, 45.0]           0.0    21
2553  2013-12-29  winter      42.0134  (40.0, 45.0]           0.0    22
2553  2013-12-29  winter      42.0134  (40.0, 45.0]           0.0    23

[61296 rows x 6 columns]
